In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns

import scipy.stats as stats
from scipy.stats import spearmanr, hypergeom

from statsmodels.stats.multitest import multipletests

## Interactive visualization
import plotly.express as px

## CellOracle
import celloracle as co
from celloracle.applications import Pseudotime_calculator

## Load data

In [2]:
## DATA (CellOracle object before fitting GRN)
adata = sc.read_h5ad("../data/data_diff_express_lncRNA.h5ad")

# Diffusion map for pseudotime calculation
sc.tl.diffmap(adata, random_state=12)

pt = Pseudotime_calculator(adata=adata,
    obsm_key="X_umap", # Dimensional reduction data name
    cluster_column_name="leiden_annotated" # Clustering data name
    )

## Pseudotime calculation

In [3]:
# Check clustering unit
print("Clustering name: ", pt.cluster_column_name)
print("Cluster list", pt.cluster_list)

Clustering name:  leiden_annotated
Cluster list ['ASO_KO', 'Ectoderm_1', 'Ectoderm_2', 'Mesoderm_1', 'Mesoderm_2', 'mESCs']


Lineage information:

=> To get better pseudotime information, calculate the pseudotime for each cell lineage individually. 

=> Then, all pseudotime information of each lineage are merged into one.

In [4]:
# Here, clusters can be classified into either MEP lineage or GMP lineage
ectoderm_lineage = ['mESCs', 'Ectoderm_1', 'Ectoderm_2', 'ASO_KO']
mesoderm_lineage = ['mESCs', 'Mesoderm_1', 'Mesoderm_2']

# Make a dictionary
lineage_dictionary = {"Lineage_ectoderm": ectoderm_lineage,
           "Lineage_mesoderm": mesoderm_lineage}

# Input lineage information into pseudotime object
pt.set_lineage(lineage_dictionary=lineage_dictionary)

# Visualize lineage information
pt.plot_lineages()
for i, num in enumerate(plt.get_fignums()):
    fig = plt.figure(num)
    fig.savefig(f"lineages_{i}.png", dpi=300, bbox_inches="tight")
    plt.close()

## Cell root visualization and selection

Interactive inspection with plotly:

In [5]:
def plot(adata, embedding_key, cluster_column_name):
    embedding = adata.obsm[embedding_key]
    df = pd.DataFrame(embedding, columns=["x", "y"])
    df["cluster"] = adata.obs[cluster_column_name].values
    df["label"] = adata.obs.index.values
    fig = px.scatter(df, x="x", y="y", hover_name=df["label"], color="cluster")
    fig.show()

plot(adata=pt.adata,
     embedding_key=pt.obsm_key,
     cluster_column_name=pt.cluster_column_name)

Root cell selection:

In [6]:
# Estimated root cell name for each lineage
root_cells = {"Lineage_ectoderm": "mES_ectodiff_asoNegControl_plate1_C01", "Lineage_mesoderm": "mES_ectodiff_asoNegControl_plate1_C01"}
pt.set_root_cells(root_cells=root_cells)

Visualize root cell:

In [7]:
# Check root cell and lineage
pt.plot_root_cells()
for i, num in enumerate(plt.get_fignums()):
    fig = plt.figure(num)
    fig.savefig(f"rootcells_{i}.png", dpi=300, bbox_inches="tight")
    plt.close()

## Visualization of DPT 

In [8]:
# Calculate pseudotime
pt.get_pseudotime_per_each_lineage()

# Check results
pt.plot_pseudotime(cmap="rainbow")
for i, num in enumerate(plt.get_fignums()):
    fig = plt.figure(num)
    fig.savefig(f"pseudotime_{i}.png", dpi=300, bbox_inches="tight")
    plt.close()

Inspect DPT data:

In [9]:
pt.adata.obs[["Pseudotime"]].head()

,Pseudotime
mES_ectodiff_asoC13_plate1_A01,0.204755
mES_ectodiff_asoC13_plate1_A03,0.249578
mES_ectodiff_asoC13_plate1_A05,0.622029
mES_ectodiff_asoC13_plate1_A11,0.757444
mES_ectodiff_asoC13_plate1_A13,0.644958


## Save

In [10]:
# Add calculated pseudotime data to the oracle object
adata.obs = pt.adata.obs

# Save updated anndata object
adata.write_h5ad("../data/data_pseudotime.h5ad")